# 2,712건 전수 판정 기반 구현 전 개선 분석

## tl;dr

- 동결 2,712건과 고유 `post_id` 2,712개를 전부 사용했다.
- pipeline 공개 263건 중 35건은 `KEEP`이 아니어서 제거·정정·보류가 필요했다.
- 정답 제목 누락은 339건이며, 이 중 explicit OCR title 107건과 title candidate 보유 249건이 우선 회수 후보다.
- 고위험 판정 872건 중 자동 의미 재검토 플래그가 없던 행은 678건이다.
- 이 수치는 개선 효과의 보장값이 아니라 구현·paired BEFORE/AFTER 검증의 모집단과 우선순위다.

## Context & Methods

분석 단위는 동결된 최종 행별 정본의 게시글 1건이다. 최종 판정은 독립 reviewer와 고위험 교차 검토·제3 중재 결과를 반영한 `independent_semantic_verdict`를 사용한다. 개선 전 기준은 해당 동결 런의 `pipeline.publish_allowed`, `pipeline.official_title`, 자동 구조 판정 및 dashboard action이다. 결과는 실행 중인 전체 104,502건으로 외삽하지 않는다.

### Key Assumptions

- `dashboard_action != KEEP`인 공개행은 최종 공개 결과에 개입이 필요했던 행으로 본다.
- 오류·보류 cohort는 서로 겹칠 수 있으므로 개선 후보별 건수를 합산해 전체 효과로 해석하지 않는다.
- 현재 worktree에는 일부 유사 게이트가 이미 있을 수 있으므로, 구현 여부가 아니라 이 동결 런에서 관측된 효과 격차를 개선 대상으로 본다.

## Data

### 1. 최종 정본을 읽고 전수성을 확인한다

In [1]:
import json
from collections import Counter
from pathlib import Path

REPO_ROOT = Path('/home/kimjang/llm_novel_dc_coommunity_parser')
LEDGER = REPO_ROOT / 'out/live_combined_row_review_20260715_143806_848671_2712_20260716/final_2712_row_adjudication_ledger.jsonl'
rows = [json.loads(line) for line in LEDGER.open(encoding='utf-8') if line.strip()]
verdict = lambda row: row['independent_semantic_verdict']
assert len(rows) == 2712
assert len({str(row['post_id']) for row in rows}) == 2712
{'rows': len(rows), 'unique_post_ids': len({str(row['post_id']) for row in rows})}

{'rows': 2712, 'unique_post_ids': 2712}

## Results

### 2. 최종 판정과 대시보드 조치를 재집계한다

In [2]:
verdict_counts = Counter(verdict(row)['review_verdict'] for row in rows)
action_counts = Counter(verdict(row)['dashboard_action'] for row in rows)
published = [row for row in rows if row['pipeline'].get('publish_allowed') is True]
published_intervention = [row for row in published if verdict(row)['dashboard_action'] != 'KEEP']
{
    'verdict_counts': dict(verdict_counts),
    'dashboard_action_counts': dict(action_counts),
    'published_rows': len(published),
    'published_keep': len(published) - len(published_intervention),
    'published_intervention': len(published_intervention),
    'published_intervention_rate': len(published_intervention) / len(published),
    'published_intervention_actions': dict(Counter(verdict(row)['dashboard_action'] for row in published_intervention)),
}

{'verdict_counts': {'NOISE_CORRECT': 891,
  'PASS': 230,
  'CONTEXT_ONLY_CORRECT': 719,
  'FN_TITLE_MISSING': 339,
  'EVIDENCE_INSUFFICIENT': 302,
  'MULTIWORK_HOLD_CORRECT': 91,
  'FP_NONNOVEL': 72,
  'EVIDENCE_CONFLICT': 25,
  'WRONG_WORK': 36,
  'DASHBOARD_TITLE_MISMATCH': 7},
 'dashboard_action_counts': {'NOT_APPLICABLE': 1581,
  'KEEP': 230,
  'CORRECT': 369,
  'HOLD': 452,
  'REMOVE': 80},
 'published_rows': 263,
 'published_keep': 228,
 'published_intervention': 35,
 'published_intervention_rate': 0.13307984790874525,
 'published_intervention_actions': {'HOLD': 12, 'CORRECT': 15, 'REMOVE': 8}}

### 3. 개선 후보별 관측 영향 모집단을 계산한다

In [3]:
high_risk_verdicts = {
    'FP_NONNOVEL', 'WRONG_WORK', 'FN_TITLE_MISSING',
    'DASHBOARD_TITLE_MISMATCH', 'EVIDENCE_CONFLICT',
    'EVIDENCE_INSUFFICIENT', 'MULTIWORK_HOLD_CORRECT',
}
high_risk = [row for row in rows if verdict(row)['review_verdict'] in high_risk_verdicts]
high_risk_unflagged = [row for row in high_risk if not row.get('manual_semantic_review_required')]
fn_rows = [row for row in rows if verdict(row)['review_verdict'] == 'FN_TITLE_MISSING']
safe_no_title = [
    row for row in rows
    if verdict(row)['review_verdict'] in {'NOISE_CORRECT', 'CONTEXT_ONLY_CORRECT'}
    and verdict(row)['dashboard_action'] == 'NOT_APPLICABLE'
]
safe_no_title_ultra = [
    row for row in safe_no_title
    if (row['source_evidence'].get('image_count') or 0) == 0
    and not row['source_evidence'].get('explicit_ocr_titles')
    and not row['source_evidence'].get('title_candidates')
    and (row['source_evidence'].get('comments_count') or 0) == 0
]
improvement_cohorts = {
    '공개 후 개입 필요': len(published_intervention),
    '정답 제목 누락': verdict_counts['FN_TITLE_MISSING'],
    '비소설·wrong-work': verdict_counts['FP_NONNOVEL'] + verdict_counts['WRONG_WORK'],
    '근거 충돌·부족·다중작품': verdict_counts['EVIDENCE_CONFLICT'] + verdict_counts['EVIDENCE_INSUFFICIENT'] + verdict_counts['MULTIWORK_HOLD_CORRECT'],
    '대시보드 제목 문자열 불일치': verdict_counts['DASHBOARD_TITLE_MISMATCH'],
    '자동 검토 플래그 누락 고위험': len(high_risk_unflagged),
    '초기 안전 라우팅 후보': len(safe_no_title_ultra),
}
improvement_cohorts

{'공개 후 개입 필요': 35,
 '정답 제목 누락': 339,
 '비소설·wrong-work': 108,
 '근거 충돌·부족·다중작품': 418,
 '대시보드 제목 문자열 불일치': 7,
 '자동 검토 플래그 누락 고위험': 678,
 '초기 안전 라우팅 후보': 188}

### 4. 제목 누락의 회수 근거와 자동 검토 coverage를 확인한다

In [4]:
fn_evidence = {
    'fn_total': len(fn_rows),
    'explicit_ocr_title': sum(bool(row['source_evidence'].get('explicit_ocr_titles')) for row in fn_rows),
    'title_candidates': sum(bool(row['source_evidence'].get('title_candidates')) for row in fn_rows),
    'image_present': sum((row['source_evidence'].get('image_count') or 0) > 0 for row in fn_rows),
    'comments_present': sum((row['source_evidence'].get('comments_count') or 0) > 0 for row in fn_rows),
    'high_confidence': sum(verdict(row)['confidence'] == 'high' for row in fn_rows),
    'manual_flagged': sum(bool(row.get('manual_semantic_review_required')) for row in fn_rows),
    'manual_flag_missing': sum(not row.get('manual_semantic_review_required') for row in fn_rows),
}
coverage = {
    name: {
        'total': sum(verdict(row)['review_verdict'] == name for row in rows),
        'flagged': sum(verdict(row)['review_verdict'] == name and bool(row.get('manual_semantic_review_required')) for row in rows),
    }
    for name in sorted(high_risk_verdicts)
}
{'fn_evidence': fn_evidence, 'automatic_review_coverage': coverage}

{'fn_evidence': {'fn_total': 339,
  'explicit_ocr_title': 107,
  'title_candidates': 249,
  'image_present': 226,
  'comments_present': 238,
  'high_confidence': 290,
  'manual_flagged': 105,
  'manual_flag_missing': 234},
 'automatic_review_coverage': {'DASHBOARD_TITLE_MISMATCH': {'total': 7,
   'flagged': 4},
  'EVIDENCE_CONFLICT': {'total': 25, 'flagged': 10},
  'EVIDENCE_INSUFFICIENT': {'total': 302, 'flagged': 19},
  'FN_TITLE_MISSING': {'total': 339, 'flagged': 105},
  'FP_NONNOVEL': {'total': 72, 'flagged': 29},
  'MULTIWORK_HOLD_CORRECT': {'total': 91, 'flagged': 9},
  'WRONG_WORK': {'total': 36, 'flagged': 18}}}

### 5. 전후 비교에 사용할 대표 행을 고정한다

In [5]:
example_ids = ['11714881', '11718117', '11718051', '11708442', '11682775', '11718070', '11718099', '11718096']
examples = []
for post_id in example_ids:
    row = next(row for row in rows if str(row['post_id']) == post_id)
    final = verdict(row)
    examples.append({
        'post_id': str(row['post_id']),
        'input_title': row['input_title'],
        'pipeline_title': row['pipeline'].get('official_title'),
        'final_verdict': final['review_verdict'],
        'recommended_title': final.get('recommended_title'),
        'dashboard_action': final['dashboard_action'],
        'rationale_ko': final['rationale_ko'],
    })
examples

[{'post_id': '11714881',
  'input_title': '중국 코로나 다음은 인도 니파인가...',
  'pipeline_title': None,
  'final_verdict': 'FP_NONNOVEL',
  'recommended_title': None,
  'dashboard_action': 'REMOVE',
  'rationale_ko': '제목·본문·OCR은 연합뉴스의 현실 니파바이러스 보도를 다룬다. 일반어 코로나를 소설 코로나 시대에 흡혈귀로 살아남기로 연결한 pipe 선택은 명백한 오탐이다.'},
 {'post_id': '11718117',
  'input_title': '싸우던 중에 갑자기 만담 뭐냐 ㅋㅋㅋ',
  'pipeline_title': None,
  'final_verdict': 'FN_TITLE_MISSING',
  'recommended_title': '아카데미의 피해자',
  'dashboard_action': 'CORRECT',
  'rationale_ko': 'OCR 본문 하단에 ‘아카데미의 피해자’, 작가 양파랑, 카카오페이지 회차가 함께 명시되어 있다. 다른 LLM-only 후보가 있어도 이미지의 직접 title 근거가 우선이므로 대시보드 title을 복구해야 한다.'},
 {'post_id': '11718051',
  'input_title': '양치기마법사 작가 신작 곧 시리즈 오나보네',
  'pipeline_title': '양치기 마법사',
  'final_verdict': 'WRONG_WORK',
  'recommended_title': '몰락민족의 천재 기사',
  'dashboard_action': 'CORRECT',
  'rationale_ko': '‘양치기마법사’는 작가를 부르는 기존작 참조이고, 글의 주제는 곧 시리즈로 오는 신작이다. 댓글이 ‘민족의 기사’라고 신작을 특정하며 후보의 공식 상세명은 ‘몰락민족의 천재 기사’이므로 기존작을 dashboard title로 쓰면 wrong 

## Takeaways

1. 최우선은 최종 공개 직전 evidence·primary-work·multiwork·media-domain 상태를 함께 보는 hard gate다. 공개 263건 중 35건이 최종적으로 유지되지 않았기 때문이다.
2. 정답 회수는 별도 lane이 필요하다. 누락 339건 전부에 권장 제목이 있고, explicit OCR title 107건과 title candidate 249건은 우선 회수·검증 대상이다.
3. 구조 모순 탐지만으로는 의미 오류를 충분히 잡지 못한다. 고위험 872건 중 678건에 자동 의미 재검토 플래그가 없었다.
4. 비소설·wrong-work·multiwork는 title-like 문자열 점수보다 media domain과 reference role을 먼저 확정해야 한다.
5. 초기 비용 최적화는 검증된 음성 1,580건 전체가 아니라 이미지·OCR·후보·댓글이 모두 없는 188건부터 canary로 시작해야 한다.
6. 모든 예상 효과는 같은 입력·모델·검색 상태의 paired BEFORE/AFTER에서 hard regression 0과 함께 재검증해야 한다.